In [1]:
from PreRun import PreRun, PostRun
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import make_scorer
from itertools import product
from datetime import date, datetime
from by_dates_Kfold import k_fold_split_option_a
from tqdm import tqdm
from pathlib import Path
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import make_scorer

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [3]:
def custom_pre_scorer(y_true, y_pred):
    return PostRun.custom_error(y_true, y_pred, a=1, b=2)


local_scorer = make_scorer(custom_pre_scorer, greater_is_better=False)

In [6]:
def bag_lin(system_id: int, read_path: str,
              met_or_inv, systems_cleaned: pd.DataFrame,
              streak_len: int,
              n_splits_outer: int, sample_spacing: int):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.fill_missing_hours()
    prerun_system.add_energy_features_only(daily_lags=2, remove_daily_lags_nans=True,
                                           include_last_year=True, remove_last_year_nans=True,
                                           include_hour_cyclic=True, include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    prerun_system.good_end_days_naive(streak=1)
    good_ends = prerun_system.end_days_naive.copy(deep=True)
    df = prerun_system.amended_data.copy(deep=True)
    df = df.dropna()
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_train = df.iloc[0:int(len(df)*0.8)]
    outer_cv = k_fold_split_option_a(
        df_train=df_train,
        good_ends=good_ends,
        n_splits=n_splits_outer,
        window_size=None,
        front_or_back='back',
        gap_day=True,
        return_type='index',
        sample_spacing=sample_spacing)
    param_grid = {
        'n_estimators': (7, 10, 15),
        'max_samples': (0.8, 1.0),
        'max_features': (0.8, 1.0),
    }
    col_names = [str(param_settings) for param_settings in product(
        param_grid['n_estimators'],
        param_grid['max_samples'],
        param_grid['max_features'])]
    outer_test_results = pd.DataFrame(
        np.zeros((len(outer_cv), len(col_names))),
        columns=col_names
    )
    for i, (train_ind, test_ind) in enumerate(tqdm(outer_cv)):
        df_tt = df_train.loc[train_ind]
        df_ho = df_train.loc[test_ind]
        X_tt = df_tt[my_cols]
        y_tt = df_tt['energy']
        X_ho = df_ho[my_cols]
        y_ho = df_ho['energy']
        for j, param_settings in enumerate(product(param_grid['n_estimators'],
            param_grid['max_samples'],
            param_grid['max_features'],
        )):
            my_reg = BaggingRegressor(
                estimator=LinearRegression(),
                n_estimators=param_settings[0],
                max_samples=param_settings[1],
                max_features=param_settings[2],
                n_jobs=4
            )
            my_reg.fit(X_tt, y_tt)
            y_pred = my_reg.predict(X_ho)
            val_error = PostRun.custom_error(y_pred,y_ho,1,2)
            outer_test_results.at[i, col_names[j]] = val_error
    out_folder = Path('./bag_lin_results/')
    if not out_folder.is_dir():
        out_folder.mkdir()
    outer_test_results.to_csv(f'./bag_lin_results/{system_id}_{met_or_inv}.csv', index=False)
    return outer_test_results

In [7]:
lin_bag_test_10 = bag_lin(
    10, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 1, -1, 5
)

100%|██████████| 282/282 [04:25<00:00,  1.06it/s]


In [10]:
lin_bag_test_10.mean(axis=0)

(7, 0.8, 0.8)     0.035695
(7, 0.8, 1.0)     0.034901
(7, 1.0, 0.8)     0.036032
(7, 1.0, 1.0)     0.034871
(10, 0.8, 0.8)    0.035220
(10, 0.8, 1.0)    0.034827
(10, 1.0, 0.8)    0.035503
(10, 1.0, 1.0)    0.034922
(15, 0.8, 0.8)    0.035887
(15, 0.8, 1.0)    0.034852
(15, 1.0, 0.8)    0.035485
(15, 1.0, 1.0)    0.034920
dtype: float64

In [16]:
lin_bag_test_10.std(axis=0)

(7, 0.8, 0.8)     0.028099
(7, 0.8, 1.0)     0.029031
(7, 1.0, 0.8)     0.028116
(7, 1.0, 1.0)     0.028943
(10, 0.8, 0.8)    0.027997
(10, 0.8, 1.0)    0.029042
(10, 1.0, 0.8)    0.027730
(10, 1.0, 1.0)    0.029353
(15, 0.8, 0.8)    0.028532
(15, 0.8, 1.0)    0.029124
(15, 1.0, 0.8)    0.028070
(15, 1.0, 1.0)    0.029190
dtype: float64

In [8]:
lin_bag_test_50 = bag_lin(
    50, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 1, -1, 5
)

100%|██████████| 388/388 [07:05<00:00,  1.10s/it]


In [11]:
lin_bag_test_50.mean(axis=0)

(7, 0.8, 0.8)     1.197141
(7, 0.8, 1.0)     1.190884
(7, 1.0, 0.8)     1.172803
(7, 1.0, 1.0)     1.192077
(10, 0.8, 0.8)    1.172966
(10, 0.8, 1.0)    1.193475
(10, 1.0, 0.8)    1.165044
(10, 1.0, 1.0)    1.193456
(15, 0.8, 0.8)    1.181617
(15, 0.8, 1.0)    1.194396
(15, 1.0, 0.8)    1.174425
(15, 1.0, 1.0)    1.193779
dtype: float64

In [17]:
lin_bag_test_50.std(axis=0)

(7, 0.8, 0.8)     1.244096
(7, 0.8, 1.0)     1.278720
(7, 1.0, 0.8)     1.206627
(7, 1.0, 1.0)     1.289715
(10, 0.8, 0.8)    1.147589
(10, 0.8, 1.0)    1.287947
(10, 1.0, 0.8)    1.152612
(10, 1.0, 1.0)    1.289219
(15, 0.8, 0.8)    1.169642
(15, 0.8, 1.0)    1.288894
(15, 1.0, 0.8)    1.184941
(15, 1.0, 1.0)    1.288549
dtype: float64

In [9]:
lin_bag_test_51 = bag_lin(
    51, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 1, -1, 5
)

100%|██████████| 395/395 [07:39<00:00,  1.16s/it]


In [12]:
lin_bag_test_51.mean(axis=0)

(7, 0.8, 0.8)     1.075244
(7, 0.8, 1.0)     1.046500
(7, 1.0, 0.8)     1.070211
(7, 1.0, 1.0)     1.047105
(10, 0.8, 0.8)    1.066466
(10, 0.8, 1.0)    1.047111
(10, 1.0, 0.8)    1.063052
(10, 1.0, 1.0)    1.046984
(15, 0.8, 0.8)    1.062712
(15, 0.8, 1.0)    1.046732
(15, 1.0, 0.8)    1.059736
(15, 1.0, 1.0)    1.047192
dtype: float64

In [18]:
lin_bag_test_51.std(axis=0)

(7, 0.8, 0.8)     0.913036
(7, 0.8, 1.0)     0.898872
(7, 1.0, 0.8)     0.891078
(7, 1.0, 1.0)     0.899634
(10, 0.8, 0.8)    0.901824
(10, 0.8, 1.0)    0.900878
(10, 1.0, 0.8)    0.890555
(10, 1.0, 1.0)    0.896640
(15, 0.8, 0.8)    0.897591
(15, 0.8, 1.0)    0.898798
(15, 1.0, 0.8)    0.897615
(15, 1.0, 1.0)    0.898862
dtype: float64